# 📊 Mini Project 3B — Pekan 4B: Tabel 4.4e, Uji Statistik 4.5, dan Visualisasi 4.6

Notebook ini mengisi bagian yang belum ada di `pekan4_experiment.ipynb` sesuai spesifikasi dokumen skenario.

| Sel | Isi |
|---|---|
| 1 | Setup & hitung estimated gas savings per kontrak |
| 2 | Tabel 4.4e — Head-to-Head vs Slither (format dokumen) |
| 3 | 4.5a — Wilcoxon signed-rank (per contract, n=46) |
| 4 | 4.5b — Kruskal-Wallis (gas savings × 5 domains) |
| 5 | 4.5c — McNemar & Cohen's Kappa (dengan catatan keterbatasan) |
| 6 | 4.6 — Visualisasi (bar chart, Venn, heatmap, scatter) |
| 7 | Ringkasan semua uji statistik & export JSON |

---
## ⚙️ Sel 1 — Setup & Estimated Gas Savings per Contract

In [1]:
import sys, json, csv, timeit
import numpy as np
from pathlib import Path
from scipy import stats

# Install visualisasi libs jika belum ada
import subprocess
for pkg in ['matplotlib', 'seaborn']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=True)

import matplotlib
matplotlib.use('Agg')   # non-interactive backend (aman di semua OS)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
print(f'matplotlib {matplotlib.__version__}, seaborn {sns.__version__}')

ROOT        = Path().resolve().parent
RESULTS_DIR = ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.detectors import ALL_DETECTORS
DETECTOR_NAMES = [n for n, _ in ALL_DETECTORS]
DOMAINS        = ['DeFi', 'NFT', 'Token', 'Governance', 'Utility']

# ── Load data
with open(RESULTS_DIR / 'pekan2_detector_results.json') as f:
    p2_results = json.load(f)
with open(RESULTS_DIR / 'pekan3_gas_benchmark.json') as f:
    bench = json.load(f)
with open(ROOT / 'contracts_metadata.json') as f:
    metadata_list = json.load(f)
try:
    with open(RESULTS_DIR / 'pekan3_slither_results.json') as f:
        slither_results = json.load(f)
except FileNotFoundError:
    slither_results = {}

# Buat map nama -> absolute file path dari contracts_metadata.json
meta_map = {m['nama']: m for m in metadata_list}

compiled = [r for r in p2_results if r['compile_ok']]

# Enrich dengan absolute file path
CONTRACTS_DIR = ROOT / 'contracts_dataset'
for r in compiled:
    if r['nama'] in meta_map:
        r['file_abs'] = meta_map[r['nama']]['file']
    else:
        # fallback: konstruksi dari filename di field 'file'
        r['file_abs'] = str(CONTRACTS_DIR / Path(r['file']).name)

# ── Gas diff per pattern (dari benchmark Pekan 3)
GAS_DIFF  = {r['pattern']: r['diff_gas']  for r in bench}
GAS_BOROS = {r['pattern']: r['boros_gas'] for r in bench}
GAS_HEMAT = {r['pattern']: r['hemat_gas'] for r in bench}

# ── Estimated savings per contract
# savings = Σ(count_det × diff_gas_det)  untuk semua detektor
est_savings = {}
for r in compiled:
    s = sum(r.get(det, 0) * GAS_DIFF.get(det, 0) for det in DETECTOR_NAMES)
    est_savings[r['nama']] = s

savings_list = [est_savings[r['nama']] for r in compiled]

n_with_savings = sum(1 for s in savings_list if s > 0)
print(f'✅ Setup selesai')
print(f'   Kontrak valid              : {len(compiled)}')
print(f'   Kontrak dengan savings > 0 : {n_with_savings}')
print(f'   Median estimated savings   : {np.median(savings_list):,.0f} gas')
print(f'   Max estimated savings      : {max(savings_list):,.0f} gas ({max(compiled, key=lambda r: est_savings[r["nama"]])["nama"]})')
print(f'\n   GAS_DIFF per pattern:')
for det in DETECTOR_NAMES:
    print(f'   {det:<28}: {GAS_DIFF.get(det, 0):>8,} gas')

matplotlib 3.10.9, seaborn 0.13.2
✅ Setup selesai
   Kontrak valid              : 46
   Kontrak dengan savings > 0 : 35
   Median estimated savings   : 12,258 gas
   Max estimated savings      : 99,210 gas (Land)

   GAS_DIFF per pattern:
   redundant_sload             :      186 gas
   unoptimized_loop            :    1,031 gas
   string_vs_bytes32           :      950 gas
   public_vs_external          :    2,673 gas
   unchecked_arithmetic        :   12,045 gas
   dead_code                   :        0 gas


---
## 🔍 Sel 2 — Tabel 4.4e: Head-to-Head vs Slither

In [2]:
from src.ast_parser import generate_ast

# ── Ukur avg analysis time (our tool) pada 5 kontrak sampel
sample5 = compiled[:5]

def _run_our_tool(meta):
    ast = generate_ast(meta['file_abs'])   # gunakan absolute path
    if ast:
        for _, detect in ALL_DETECTORS:
            detect(ast)

our_times = []
for m in sample5:
    t = timeit.timeit(lambda: _run_our_tool(m), number=1)
    our_times.append(t)
    print(f'  Our tool — {m["nama"]}: {t:.2f}s')

avg_our_time = np.mean(our_times)

# ── Ukur avg analysis time (Slither) pada 1 kontrak yang bisa (0.8.x era)
import shutil
import tempfile
slither_exe = shutil.which('slither')
avg_slither_time_str = 'N/A (gagal pada kontrak 0.4.x)'

if slither_exe:
    # Cari kontrak 0.8.x yang ada (NFT modern biasanya 0.8.x)
    modern = [m for m in compiled if m['domain'] == 'NFT' and m['complexity'] == 'Complex'][:1]
    if modern:
        import subprocess as sp
        t0 = timeit.default_timer()
        with tempfile.NamedTemporaryFile(suffix='.json', delete=False) as tmp:
            tmp_path = tmp.name
        sp.run([slither_exe, modern[0]['file_abs'], '--json', tmp_path, '--disable-color'],
               capture_output=True, text=True, timeout=120)
        Path(tmp_path).unlink(missing_ok=True)
        t1 = timeit.default_timer()
        avg_slither_time_str = f'{t1-t0:.1f}s (1 kontrak: {modern[0]["nama"]})'
        print(f'  Slither — {modern[0]["nama"]}: {t1-t0:.2f}s')

# ── Hitung metrik perbandingan
our_total = sum(sum(r.get(d, 0) for d in DETECTOR_NAMES) for r in compiled)

GAS_DETECTORS_SLITHER = {'costly-loop', 'dead-code', 'unused-return',
                         'cache-array-length', 'storage-array', 'redundant-statements'}
slither_total = sum(
    len([x for x in v.get('all_detectors', []) if x in GAS_DETECTORS_SLITHER])
    for v in slither_results.values()
)
overlap = 0   # Slither found 0 on all sampled contracts

# ── Tampilkan tabel
print('\n=== TABEL 4.4e: HEAD-TO-HEAD vs SLITHER (50 CONTRACTS) ===\n')
rows = [
    ('Total Patterns Detected',     str(our_total),           str(slither_total),   str(overlap)),
    ('Unique to Our Tool',          str(our_total - overlap), '—',                  '—'),
    ('Unique to Slither',           '—',                      str(slither_total - overlap), '—'),
    ('Precision (shared patterns)', 'N/A*',                   'N/A*',               '—'),
    ('Gas Quantification',          'Ya (per pattern)',        'Tidak',              '—'),
    ('Avg Analysis Time',           f'{avg_our_time:.2f}s/kontrak', avg_slither_time_str, '—'),
]
print(f'{"Metrik":<30} {"Our Tool":<30} {"Slither":<35} {"Overlap":<10}')
print('─' * 108)
for r in rows:
    print(f'{r[0]:<30} {r[1]:<30} {r[2]:<35} {r[3]:<10}')

print('\n*Precision tidak dapat dihitung tanpa ground truth dari audit manual')
print('**Slither 0.11.5 tidak mendukung pragma solidity 0.4.x (mayoritas dataset)')
print(f'**Slither diuji pada {len(slither_results)} kontrak sampel, semua dari domain DeFi (0.4.x–0.6.x era)')

# ── Simpan CSV
csv_4e = RESULTS_DIR / 'tabel_4_4e_headtohead.csv'
with open(csv_4e, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Metrik', 'Our Tool', 'Slither', 'Overlap'])
    w.writerows(rows)
print(f'\n💾 {csv_4e}')

  Our tool — WETH9: 0.05s


  Our tool — UniswapV2Router02: 0.23s


  Our tool — Dai: 0.07s
  Our tool — FiatTokenProxy: 0.05s


  Our tool — Uni: 0.12s


  Slither — MutantApeYachtClub: 1.22s

=== TABEL 4.4e: HEAD-TO-HEAD vs SLITHER (50 CONTRACTS) ===

Metrik                         Our Tool                       Slither                             Overlap   
────────────────────────────────────────────────────────────────────────────────────────────────────────────
Total Patterns Detected        646                            0                                   0         
Unique to Our Tool             646                            —                                   —         
Unique to Slither              —                              0                                   —         
Precision (shared patterns)    N/A*                           N/A*                                —         
Gas Quantification             Ya (per pattern)               Tidak                               —         
Avg Analysis Time              0.10s/kontrak                  1.2s (1 kontrak: MutantApeYachtClub) —         

*Precision tidak dapat dihi

---
## 🧪 Sel 3 — 4.5a: Wilcoxon Signed-Rank (Per Contract, n=46)

In [3]:
from scipy.stats import wilcoxon

print('=== 4.5a: WILCOXON SIGNED-RANK (PER CONTRACT) ===')
print('H0 : Median estimated gas savings = 0')
print('H1 : Median estimated gas savings > 0')
print(f'n  : {len(savings_list)} kontrak (unit observasi = kontrak, bukan pattern)\n')

print('Metode estimasi savings per kontrak:')
print('  savings[i] = Σ (count_detector[i][d] × avg_diff_gas[d]) untuk d = 6 pattern')
print('  avg_diff_gas berasal dari benchmark Hardhat (Pekan 3)\n')

# Distribusi savings
s_arr = np.array(savings_list)
print(f'Distribusi estimated savings (gas):')
print(f'  Min    : {s_arr.min():>10,.0f}')
print(f'  Median : {np.median(s_arr):>10,.0f}')
print(f'  Mean   : {s_arr.mean():>10,.0f}')
print(f'  Max    : {s_arr.max():>10,.0f}')
print(f'  > 0    : {(s_arr > 0).sum():>10} kontrak')
print(f'  = 0    : {(s_arr == 0).sum():>10} kontrak (dikecualikan dari uji)')

W_contract, p_contract = np.nan, np.nan
try:
    W_contract, p_contract = wilcoxon(s_arr, alternative='greater', zero_method='wilcox')
    print(f'\nWilcoxon W = {W_contract:.1f}')
    print(f'p-value    = {p_contract:.6f}')
    sig = p_contract < 0.05
    print(f'\nKeputusan (α=0.05): {"✅ H1 DITERIMA — savings signifikan" if sig else "❌ H1 ditolak"}')
except Exception as e:
    print(f'Error: {e}')

print('\nCatatan: Wilcoxon per-contract lebih kuat secara statistik (n=46) dibanding')
print('per-pattern (n=6) yang dilakukan di pekan4_experiment.ipynb.')

=== 4.5a: WILCOXON SIGNED-RANK (PER CONTRACT) ===
H0 : Median estimated gas savings = 0
H1 : Median estimated gas savings > 0
n  : 46 kontrak (unit observasi = kontrak, bukan pattern)

Metode estimasi savings per kontrak:
  savings[i] = Σ (count_detector[i][d] × avg_diff_gas[d]) untuk d = 6 pattern
  avg_diff_gas berasal dari benchmark Hardhat (Pekan 3)

Distribusi estimated savings (gas):
  Min    :          0
  Median :     12,258
  Mean   :     22,046
  Max    :     99,210
  > 0    :         35 kontrak
  = 0    :         11 kontrak (dikecualikan dari uji)

Wilcoxon W = 630.0
p-value    = 0.000000

Keputusan (α=0.05): ✅ H1 DITERIMA — savings signifikan

Catatan: Wilcoxon per-contract lebih kuat secara statistik (n=46) dibanding
per-pattern (n=6) yang dilakukan di pekan4_experiment.ipynb.


---
## 📊 Sel 4 — 4.5b: Kruskal-Wallis (Gas Savings × 5 Domains)

In [4]:
from scipy.stats import kruskal

print('=== 4.5b: KRUSKAL-WALLIS (GAS SAVINGS × DOMAIN) ===')
print('H0 : Distribusi estimated gas savings sama di semua domain')
print('H1 : Setidaknya satu domain memiliki distribusi berbeda\n')

domain_savings = {}
for domain in DOMAINS:
    vals = [est_savings[r['nama']] for r in compiled if r['domain'] == domain]
    domain_savings[domain] = vals

print(f'{"Domain":<14} {"n":>4} {"Median savings":>16} {"Mean savings":>14} {"Max savings":>13}')
print('─' * 65)
for domain, vals in domain_savings.items():
    print(f'{domain:<14} {len(vals):>4} {np.median(vals):>16,.0f} {np.mean(vals):>14,.0f} {max(vals):>13,.0f}')

H_domain, p_domain = np.nan, np.nan
valid_groups = [v for v in domain_savings.values() if len(v) >= 2]
if len(valid_groups) >= 2:
    try:
        H_domain, p_domain = kruskal(*valid_groups)
        print(f'\nKruskal-Wallis H = {H_domain:.4f}, p = {p_domain:.4f}')
        sig = p_domain < 0.05
        print(f'Keputusan (α=0.05): '
              f'{"H0 DITOLAK — domain berpengaruh signifikan" if sig else "H0 diterima — tidak ada perbedaan signifikan antar domain"}')
    except Exception as e:
        print(f'Error: {e}')

print('\nCatatan: Berbeda dari pekan4 yang menguji complexity level vs jumlah findings.')
print('Uji ini menguji apakah BESARNYA potensi penghematan gas berbeda per domain.')

=== 4.5b: KRUSKAL-WALLIS (GAS SAVINGS × DOMAIN) ===
H0 : Distribusi estimated gas savings sama di semua domain
H1 : Setidaknya satu domain memiliki distribusi berbeda

Domain            n   Median savings   Mean savings   Max savings
─────────────────────────────────────────────────────────────────
DeFi             10           14,272         18,793        58,737
NFT               9            5,346         23,219        99,210
Token             9           49,044         47,241        89,081
Governance        9           13,336         16,910        38,892
Utility           9                0          4,426        33,931

Kruskal-Wallis H = 14.0149, p = 0.0072
Keputusan (α=0.05): H0 DITOLAK — domain berpengaruh signifikan

Catatan: Berbeda dari pekan4 yang menguji complexity level vs jumlah findings.
Uji ini menguji apakah BESARNYA potensi penghematan gas berbeda per domain.


---
## 📋 Sel 5 — 4.5c: McNemar, Cohen's Kappa & Spearman

In [5]:
from scipy.stats import spearmanr, binomtest

# ── McNemar: our tool vs Slither (binary per contract)
print('=== McNemar Test: Our Tool vs Slither ===')
print('Binary: apakah kontrak memiliki gas-related findings? (Ya=1, Tidak=0)\n')

GAS_DETECTORS_SLITHER = {'costly-loop','dead-code','unused-return',
                         'cache-array-length','storage-array','redundant-statements'}

mc_p = np.nan
a = b = c = d_mc = 0

if slither_results:
    for nama, sdata in slither_results.items():
        our  = sum(r.get(det,0) for r in compiled if r['nama']==nama
                   for det in DETECTOR_NAMES) > 0
        slit = len([x for x in sdata.get('all_detectors',[]) if x in GAS_DETECTORS_SLITHER]) > 0
        if our and slit:   a += 1
        elif our:          b += 1
        elif slit:         c += 1
        else:              d_mc += 1

    print(f'Tabel 2x2 ({len(slither_results)} kontrak sampel):')
    print(f'  a (keduanya menemukan) = {a}')
    print(f'  b (hanya our tool)     = {b}')
    print(f'  c (hanya Slither)      = {c}')
    print(f'  d (keduanya tidak)     = {d_mc}')

    # Exact McNemar — bisa dihitung selama b+c > 0
    if b + c > 0:
        mc = binomtest(min(b, c), b + c, 0.5, alternative='two-sided')
        mc_p = mc.pvalue
        print(f'\nMcNemar exact test:')
        print(f'  H0: P(deteksi eksklusif our tool) = P(deteksi eksklusif Slither)')
        print(f'  Discordant pairs: b={b}, c={c}  →  p = {mc_p:.5f}')
        print(f'  Keputusan (a=0.05): {"H0 DITOLAK" if mc_p < 0.05 else "H0 diterima"}')
        print(f'\n  Catatan: p={mc_p:.4f} menunjukkan our tool mendeteksi secara signifikan lebih banyak.')
        print(f'  Hasil ini dipengaruhi keterbatasan Slither pada pragma 0.4.x, bukan superioritas murni.')
    else:
        print('\n  McNemar: b+c=0, tidak ada pasangan diskordansi.')

# ── Cohen's Kappa — dihitung dari tabel 2x2 tanpa ground truth eksternal
print('\n=== Cohens Kappa: Inter-Rater Agreement (Our Tool vs Slither) ===')
print('Formula: k = (Po - Pe) / (1 - Pe)\n')

kappa = np.nan
n_mc = len(slither_results) if slither_results else 0

if n_mc > 0:
    Po = (a + d_mc) / n_mc
    p_pos_ours  = (a + b) / n_mc
    p_pos_slith = (a + c) / n_mc
    p_neg_ours  = (c + d_mc) / n_mc
    p_neg_slith = (b + d_mc) / n_mc
    Pe = (p_pos_ours * p_pos_slith) + (p_neg_ours * p_neg_slith)
    kappa = (Po - Pe) / (1 - Pe) if (1 - Pe) != 0 else 0.0

    print(f'  n   = {n_mc}  |  a={a}, b={b}, c={c}, d={d_mc}')
    print(f'  Po  = (a+d)/n = ({a}+{d_mc})/{n_mc} = {Po:.5f}')
    print(f'  Pe  = {p_pos_ours:.2f}*{p_pos_slith:.2f} + {p_neg_ours:.2f}*{p_neg_slith:.2f} = {Pe:.5f}')
    print(f'  k   = ({Po:.5f} - {Pe:.5f}) / (1 - {Pe:.5f}) = {kappa:.5f}')
    print()

    if   kappa < 0:    interp = 'Kesepakatan lebih buruk dari kebetulan'
    elif kappa == 0.0: interp = 'Kesepakatan setara kebetulan (no agreement)'
    elif kappa < 0.20: interp = 'Kesepakatan sangat lemah (slight)'
    elif kappa < 0.40: interp = 'Kesepakatan lemah (fair)'
    elif kappa < 0.60: interp = 'Kesepakatan sedang (moderate)'
    else:              interp = 'Kesepakatan kuat (substantial/almost perfect)'
    print(f'  Interpretasi: {interp}')
    print()
    print(f'  Konteks: k=0.0 karena Slither selalu memberi label "tidak ada findings"')
    print(f'  pada semua 10 sampel (keterbatasan kompatibilitas solc 0.4.x).')
    print(f'  Nilai ini valid dan dapat dilaporkan dalam thesis.')

# ── Spearman (recap dari pekan4)
print('\n=== Spearman Correlation: LOC vs Total Patterns ===')
locs   = [r['loc'] for r in compiled]
totals = [sum(r.get(d,0) for d in DETECTOR_NAMES) for r in compiled]
rho, p_rho = spearmanr(locs, totals)
print(f'rho = {rho:.5f}, p = {p_rho:.5f}')
print(f'Interpretasi: {"korelasi signifikan" if p_rho < 0.05 else "tidak signifikan"}, '
      f'arah {"positif" if rho > 0 else "negatif"} lemah')

=== McNemar Test: Our Tool vs Slither ===
Binary: apakah kontrak memiliki gas-related findings? (Ya=1, Tidak=0)

Tabel 2x2 (10 kontrak sampel):
  a (keduanya menemukan) = 0
  b (hanya our tool)     = 8
  c (hanya Slither)      = 0
  d (keduanya tidak)     = 2

McNemar exact test:
  H0: P(deteksi eksklusif our tool) = P(deteksi eksklusif Slither)
  Discordant pairs: b=8, c=0  →  p = 0.00781
  Keputusan (a=0.05): H0 DITOLAK

  Catatan: p=0.0078 menunjukkan our tool mendeteksi secara signifikan lebih banyak.
  Hasil ini dipengaruhi keterbatasan Slither pada pragma 0.4.x, bukan superioritas murni.

=== Cohens Kappa: Inter-Rater Agreement (Our Tool vs Slither) ===
Formula: k = (Po - Pe) / (1 - Pe)

  n   = 10  |  a=0, b=8, c=0, d=2
  Po  = (a+d)/n = (0+2)/10 = 0.20000
  Pe  = 0.80*0.00 + 0.20*1.00 = 0.20000
  k   = (0.20000 - 0.20000) / (1 - 0.20000) = 0.00000

  Interpretasi: Kesepakatan setara kebetulan (no agreement)

  Konteks: k=0.0 karena Slither selalu memberi label "tidak ada findin

---
## 🎨 Sel 6 — 4.6: Visualisasi (4 Chart)

In [6]:
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'
})

# ── Chart 1: Bar chart — Gas Savings per Pattern
fig1, ax1 = plt.subplots(figsize=(9, 5))
patterns  = [r['pattern'] for r in bench]
pct_saves = [r['pct_save'] for r in bench]
colors    = ['#2ecc71' if p > 0 else '#bdc3c7' for p in pct_saves]
bars = ax1.bar(range(len(patterns)), pct_saves, color=colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, pct_saves):
    if val > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_title('Gas Savings per Anti-Pattern (Benchmark, optimizer=off)', pad=12)
ax1.set_ylabel('Penghematan Gas (%)')
ax1.set_xlabel('Anti-Pattern')
ax1.set_xticks(range(len(patterns)))
ax1.set_xticklabels(patterns, rotation=20, ha='right')
ax1.set_ylim(0, max(pct_saves) * 1.18)
ax1.axhline(0, color='black', linewidth=0.7)
green_patch = mpatches.Patch(color='#2ecc71', label='Penghematan positif')
grey_patch  = mpatches.Patch(color='#bdc3c7', label='Tidak ada penghematan')
ax1.legend(handles=[green_patch, grey_patch], loc='upper left')
ax1.grid(axis='y', alpha=0.35)
fig1.tight_layout()
p1 = FIGURES_DIR / 'chart1_gas_savings_per_pattern.png'
fig1.savefig(p1)
plt.close(fig1)
print(f'💾 {p1}')

# ── Chart 2: Venn Diagram — Our Tool vs Slither
fig2, ax2 = plt.subplots(figsize=(7, 5))
our_n     = sum(1 for r in compiled if sum(r.get(d,0) for d in DETECTOR_NAMES) > 0)
slith_n   = sum(1 for v in slither_results.values()
                if any(x in GAS_DETECTORS_SLITHER for x in v.get('all_detectors',[])))
circle_our   = plt.Circle((0.37, 0.5), 0.30, color='#3498db', alpha=0.45, label=f'Our Tool ({our_n})')
circle_slith = plt.Circle((0.63, 0.5), 0.30, color='#e74c3c', alpha=0.45, label=f'Slither ({slith_n})')
ax2.add_patch(circle_our)
ax2.add_patch(circle_slith)
ax2.text(0.27, 0.50, str(our_n), ha='center', va='center', fontsize=18, fontweight='bold', color='#1a5276')
ax2.text(0.73, 0.50, str(slith_n), ha='center', va='center', fontsize=18, fontweight='bold', color='#922b21')
ax2.text(0.50, 0.50, '0', ha='center', va='center', fontsize=16, color='#555')
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
ax2.set_aspect('equal'); ax2.axis('off')
ax2.set_title('Venn Diagram: Kontrak dengan Gas Findings\n(Our Tool vs Slither, 10 sampel)', pad=12)
ax2.text(0.50, 0.10,
         '*Slither 0 findings: ketidakkompatibilan pragma solidity 0.4.x',
         ha='center', va='center', fontsize=9, color='#777', style='italic')
ax2.legend(handles=[circle_our, circle_slith], loc='upper center',
           bbox_to_anchor=(0.5, -0.02), ncol=2)
fig2.tight_layout()
p2 = FIGURES_DIR / 'chart2_venn_our_vs_slither.png'
fig2.savefig(p2)
plt.close(fig2)
print(f'💾 {p2}')

# ── Chart 3: Heatmap — Pattern Frequency per Domain
SHORT_DET = ['rs', 'ul', 'sb', 'pe', 'ua', 'dc']
matrix = []
for domain in DOMAINS:
    row = [sum(r.get(det,0) for r in compiled if r['domain']==domain)
           for det in DETECTOR_NAMES]
    matrix.append(row)
mat = np.array(matrix)

fig3, ax3 = plt.subplots(figsize=(9, 4.5))
im = ax3.imshow(mat, cmap='YlOrRd', aspect='auto')
ax3.set_xticks(range(len(DETECTOR_NAMES)))
ax3.set_xticklabels([d.replace('_', '\n') for d in DETECTOR_NAMES], fontsize=9)
ax3.set_yticks(range(len(DOMAINS)))
ax3.set_yticklabels(DOMAINS)
for i in range(len(DOMAINS)):
    for j in range(len(DETECTOR_NAMES)):
        v = mat[i, j]
        ax3.text(j, i, str(v), ha='center', va='center',
                 fontsize=11, fontweight='bold',
                 color='white' if v > mat.max()*0.6 else 'black')
plt.colorbar(im, ax=ax3, label='Jumlah Findings')
ax3.set_title('Heatmap: Frekuensi Anti-Pattern per Domain', pad=12)
fig3.tight_layout()
p3 = FIGURES_DIR / 'chart3_heatmap_domain_pattern.png'
fig3.savefig(p3)
plt.close(fig3)
print(f'💾 {p3}')

# ── Chart 4: Scatter — LOC vs Total Findings
DOMAIN_COLORS = {'DeFi':'#3498db','NFT':'#e74c3c','Token':'#2ecc71',
                 'Governance':'#f39c12','Utility':'#9b59b6'}
fig4, ax4 = plt.subplots(figsize=(8, 5.5))
for domain in DOMAINS:
    dr = [r for r in compiled if r['domain'] == domain]
    xs = [r['loc'] for r in dr]
    ys = [sum(r.get(d,0) for d in DETECTOR_NAMES) for r in dr]
    ax4.scatter(xs, ys, c=DOMAIN_COLORS[domain], label=domain, alpha=0.75, s=60, edgecolors='white', linewidth=0.5)

all_locs   = [r['loc'] for r in compiled]
all_totals = [sum(r.get(d,0) for d in DETECTOR_NAMES) for r in compiled]
z = np.polyfit(all_locs, all_totals, 1)
p_line = np.poly1d(z)
x_line = np.linspace(min(all_locs), max(all_locs), 100)
ax4.plot(x_line, p_line(x_line), 'k--', alpha=0.4, linewidth=1.2, label='Trend (linear fit)')

ax4.set_xlabel('Lines of Code (LOC)')
ax4.set_ylabel('Total Findings')
ax4.set_title(f'Scatter: LOC vs Total Anti-Pattern Findings\n(Spearman ρ = {rho:.3f}, p = {p_rho:.3f})', pad=12)
ax4.legend(loc='upper right', fontsize=9)
ax4.grid(alpha=0.25)
fig4.tight_layout()
p4 = FIGURES_DIR / 'chart4_scatter_loc_findings.png'
fig4.savefig(p4)
plt.close(fig4)
print(f'💾 {p4}')

print('\n✅ Semua 4 chart berhasil disimpan ke results/figures/')

💾 C:\Users\Ridho\project\KBJ\gas_optimization\results\figures\chart1_gas_savings_per_pattern.png


💾 C:\Users\Ridho\project\KBJ\gas_optimization\results\figures\chart2_venn_our_vs_slither.png


💾 C:\Users\Ridho\project\KBJ\gas_optimization\results\figures\chart3_heatmap_domain_pattern.png


💾 C:\Users\Ridho\project\KBJ\gas_optimization\results\figures\chart4_scatter_loc_findings.png

✅ Semua 4 chart berhasil disimpan ke results/figures/


---
## 📋 Sel 7 — Ringkasan Semua Uji Statistik & Export

In [7]:
def _safe(v):
    return None if (v is None or (isinstance(v, float) and np.isnan(v))) else round(float(v), 6)

# Load hasil dari pekan4 untuk gabung
try:
    with open(RESULTS_DIR / 'pekan4_statistical_tests.json') as f:
        existing = json.load(f)
except FileNotFoundError:
    existing = {}

# Tambah/update dengan hasil baru
existing.update({
    'wilcoxon_per_contract': {
        'W': _safe(W_contract), 'p': _safe(p_contract),
        'n': len([s for s in savings_list if s != 0]),
        'unit': 'per kontrak (estimated savings)',
        'significant_005': bool(not np.isnan(p_contract) and p_contract < 0.05)
    },
    'kruskal_wallis_per_domain': {
        'H': _safe(H_domain), 'p': _safe(p_domain),
        'groups': DOMAINS,
        'unit': 'estimated gas savings per domain',
        'significant_005': bool(not np.isnan(p_domain) and p_domain < 0.05)
    },
    'mcnemar_our_vs_slither': {
        'a': a, 'b': b, 'c': c, 'd': d_mc,
        'p': _safe(mc_p),
        'n_sample': len(slither_results),
        'significant_005': bool(not np.isnan(mc_p) and mc_p < 0.05),
        'note': 'Exact McNemar (binomial). Hasil dipengaruhi keterbatasan Slither pada solc 0.4.x'
    },
    'cohens_kappa': {
        'value': _safe(kappa),
        'n_sample': len(slither_results),
        'note': 'Dihitung dari 2x2 contingency table (inter-rater: our tool vs Slither). k=0 karena Slither tidak dapat analisis solc 0.4.x'
    },
    'spearman_loc_findings': {
        'rho': _safe(rho), 'p': _safe(p_rho),
        'significant_005': bool(p_rho < 0.05)
    },
})

stat_path = RESULTS_DIR / 'pekan4_statistical_tests.json'
with open(stat_path, 'w') as f:
    json.dump(existing, f, indent=2)

# ── Tabel ringkasan semua uji (tampilan 5 desimal)
ex_w  = existing.get('wilcoxon_gas_savings', {})
ex_kw = existing.get('kruskal_wallis_complexity', {})
ex_cs = existing.get('chi_square_domain', {})

mc_p_disp    = f'{float(mc_p):.5f}'  if not np.isnan(mc_p)  else 'N/A'
mc_sig_disp  = ('✅ Ya*' if not np.isnan(mc_p) and mc_p < 0.05 else '❌ Tidak')
kappa_disp   = f'{float(kappa):.5f}' if not np.isnan(kappa) else 'N/A'

print('=== RINGKASAN SEMUA UJI STATISTIK ===')
print(f'\n{"Uji":<40} {"Statistik":>14} {"p-value":>10} {"Signifikan":>12}  Kesimpulan')
print('─' * 110)
summary_rows = [
    ('Wilcoxon (per-pattern, n=6)',
     f'W={float(ex_w.get("W",0)):.5f}',
     f'{float(ex_w.get("p",0)):.5f}',
     '✅ Ya', 'Gas savings signifikan'),
    (f'Wilcoxon (per-contract, n={len([s for s in savings_list if s!=0])})',
     f'W={float(W_contract):.5f}',
     f'{float(p_contract):.5f}',
     ('✅ Ya' if not np.isnan(p_contract) and p_contract < 0.05 else '❌ Tidak'),
     'Savings per kontrak'),
    ('Kruskal-Wallis (complexity vs findings)',
     f'H={float(ex_kw.get("H",0)):.5f}',
     f'{float(ex_kw.get("p",0)):.5f}',
     '❌ Tidak', 'Complexity tdk berpengaruh'),
    ('Kruskal-Wallis (domain vs savings)',
     f'H={float(H_domain):.5f}',
     f'{float(p_domain):.5f}',
     ('✅ Ya' if not np.isnan(p_domain) and p_domain < 0.05 else '❌ Tidak'),
     'Gas savings per domain'),
    ('Chi-square (domain vs findings)',
     f'χ²={float(ex_cs.get("chi2",0)):.5f}',
     f'{float(ex_cs.get("p",0)):.5f}',
     '❌ Tidak', 'Domain tdk berpengaruh'),
    ('Spearman (LOC vs findings)',
     f'ρ={float(rho):.5f}',
     f'{float(p_rho):.5f}',
     '❌ Tidak', 'LOC tdk berkorelasi'),
    ('McNemar (our tool vs Slither)',
     f'b={b}, c={c}',
     mc_p_disp,
     mc_sig_disp,
     'Our tool >> Slither (terbatas 0.4.x)'),
    ("Cohen's Kappa (our tool vs Slither)",
     f'k={kappa_disp}',
     '—',
     '—',
     'No agreement beyond chance'),
]
for row in summary_rows:
    print(f'{row[0]:<40} {str(row[1]):>14} {str(row[2]):>10} {row[3]:>12}  {row[4]}')

print('\n*McNemar signifikan karena Slither 0 findings (bukan superioritas murni)')
print(f'\n💾 {stat_path}')

# ── Checklist
print('\n' + '='*55)
print('  CHECKLIST AKHIR PEKAN 4B')
print('='*55)
checks = [
    ('Tabel 4.4e (Head-to-Head format) selesai',
        (RESULTS_DIR / 'tabel_4_4e_headtohead.csv').exists()),
    ('Wilcoxon per-contract selesai',          not np.isnan(W_contract)),
    ('Kruskal-Wallis per-domain selesai',       not np.isnan(H_domain)),
    ('McNemar dihitung (p tersedia)',           not np.isnan(mc_p)),
    ("Cohen's Kappa dihitung (k tersedia)",     not np.isnan(kappa)),
    ('Chart 1 (bar gas savings) disimpan',
        (FIGURES_DIR / 'chart1_gas_savings_per_pattern.png').exists()),
    ('Chart 2 (Venn diagram) disimpan',
        (FIGURES_DIR / 'chart2_venn_our_vs_slither.png').exists()),
    ('Chart 3 (heatmap) disimpan',
        (FIGURES_DIR / 'chart3_heatmap_domain_pattern.png').exists()),
    ('Chart 4 (scatter LOC) disimpan',
        (FIGURES_DIR / 'chart4_scatter_loc_findings.png').exists()),
    ('Statistical tests JSON diupdate',         stat_path.exists()),
]
all_ok = True
for label, ok in checks:
    if not ok: all_ok = False
    print(f'  {"✅" if ok else "❌"} {label}')
print('='*55)
print('  ✅ Pekan 4B selesai!' if all_ok else '  ⚠️  Cek item di atas')
print('='*55)

=== RINGKASAN SEMUA UJI STATISTIK ===

Uji                                           Statistik    p-value   Signifikan  Kesimpulan
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
Wilcoxon (per-pattern, n=6)                  W=15.00000    0.03125         ✅ Ya  Gas savings signifikan
Wilcoxon (per-contract, n=35)               W=630.00000    0.00000         ✅ Ya  Savings per kontrak
Kruskal-Wallis (complexity vs findings)       H=0.63106    0.42697      ❌ Tidak  Complexity tdk berpengaruh
Kruskal-Wallis (domain vs savings)           H=14.01485    0.00725         ✅ Ya  Gas savings per domain
Chi-square (domain vs findings)              χ²=8.56784    0.07286      ❌ Tidak  Domain tdk berpengaruh
Spearman (LOC vs findings)                   ρ=-0.26146    0.07923      ❌ Tidak  LOC tdk berkorelasi
McNemar (our tool vs Slither)                  b=8, c=0    0.00781        ✅ Ya*  Our tool >> Slither (terbatas 0.4.x)
Cohen's Kappa (our